# MRM-Bench baseline logistic regression, step by step

## 0. Title and how to use this notebook

**What this notebook is.** The clean logistic-regression baseline of MRM-Bench, and its only implementation. It runs
the whole pipeline one step at a time, explains each step in plain English, shows what the data look like after each
step, and ends (section 12) with a check that its results are **identical** to the official reference results in
`reference/`.

**The prediction task.** Take every loan that, at the end of a December (the prediction month $t$), is still active and
not seriously late. Estimate the probability that it becomes **90 or more days past due** at least once in the next
12 months ($t+1 \dots t+12$), using only information available at $t$. The full rules are in `CLAUDE.md`, section
"Outcome specification" (a draft, pending Dr. Ku's approval). This is "12-month 90+ DPD serious delinquency",
not a regulatory default definition.

**History: the original script.** The baseline was first written as a script, `src/baseline_logreg.py`. It is no
longer in the repository; it is kept in git history at commit `9253e97` (`git show 9253e97:src/baseline_logreg.py`).
That script produced the reference results in `reference/`. This notebook keeps its function names, constants, column
names, seed and feature lists, and section 12 proves that it reproduces those results exactly. You can experiment
here: if you change something, section 12 tells you exactly which results now differ from the reference. To make a
change official, follow the reminder at the end of section 13.

### How to use it

1. **Select the kernel (VS Code).** Click **Select Kernel** (top right of the notebook) → **Python Environments…** →
   choose the project environment, listed as **finance-ai-system** or **.venv (Python 3.12.12)**, with the path
   `.venv/bin/python`. If it is not listed, run `uv sync` in a terminal at the repository root, then run the command
   **Developer: Reload Window**. The imports cell in section 1 prints the kernel path, so you can check it.
2. **Run from top to bottom** (**Run All**). Later cells use variables made by earlier ones. After you change a
   setting, rerun from the section given in the table in section 13, or simply **Restart** and **Run All**.
3. **Run time:** about 1 minute on the development Mac (measured: 54 seconds for the whole notebook, 37 of them for the loan bootstrap in section 9); allow a few minutes on a slower machine. For a quick run,
   set `N_BOOT = 100` in the SETTINGS cell; section 12 will then report that only the confidence intervals differ.
4. **Files:** the notebook reads `data/raw/sample_2015.zip` and `reference/`. It writes the unzipped data to
   `sflld_extracted/` (gitignored) and, only when `SAVE_OUTPUTS = True` in the SETTINGS cell, five result files to
   `outputs/` (gitignored). It never writes to `reference/`.

> **Warning: real loan data.** This notebook displays real Freddie Mac records (small samples). Freddie Mac's terms
> do not allow redistributing them. **nbstripout** removes every cell output whenever the notebook is committed, so
> outputs never enter git. It does **not** clean the file on your disk: before you email, upload or share the
> `.ipynb` file itself, run **Clear All Outputs**. nbstripout is configured per clone (in `.git/config`): after a
> fresh clone, run `uv sync` and then `uv run nbstripout --install`. Check it with `uv run nbstripout --status`.

To run the whole notebook without VS Code (the executed copy is written outside the repository):

```
uv run jupyter nbconvert --to notebook --execute notebooks/01_baseline_logreg_walkthrough.ipynb --output /tmp/nb_check.ipynb
```

### Contents

| § | Section | What you learn |
|---|---|---|
| 0 | Title and how to use this notebook | this page |
| 1 | Setup and configuration | repository paths, imports, the single **SETTINGS** cell |
| 2 | Loading the raw files | the column layout, reading as text, structural checks |
| 3 | Preprocessing the origination data | "Not Available" codes, the five derived features |
| 4 | Preprocessing the monthly performance data | the month index $t$, delinquency and zero-balance codes |
| 5 | A toy example (synthetic data) | how labels and features are built, loan by loan |
| 6 | Building the real snapshots | label counts, the December 2019 spike |
| 7 | The train/test split | hash split by loan, time split, why censored rows are dropped |
| 8 | The logistic regression model | the math, the preprocessing pipeline, fitting |
| 9 | Evaluation | ROC-AUC, PR-AUC, Brier score, the loan bootstrap, calibration |
| 10 | Coefficients | how to read standardized and one-hot coefficients |
| 11 | Leakage check | future invariance (truncation and perturbation) on the toy panel; the formal tests |
| 12 | Does the notebook match the reference? | the final verdict against `reference/`; optionally saving the results |
| 13 | Where to make changes, and optional exercises | safe edit points |

## 1. Setup and configuration

**What this cell does:** Moves the working folder to the repository root (the folder that contains `pyproject.toml`).

**Why it matters:** VS Code and nbconvert start a notebook in its own folder (`notebooks/`), but the notebook's paths (`data/raw/...`, `reference/`, `outputs/`, `sflld_extracted/`) are relative to the repository root. Running from the root means the same paths work here, and the unzipped data lands in the gitignored `sflld_extracted/` folder instead of inside `notebooks/`.

**What to look for:** The printed folder ends in `Finance AI System`.

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():        # walk up until we reach the repository root
    if ROOT.parent == ROOT:
        raise FileNotFoundError("pyproject.toml not found above the current folder: open the notebook inside the repo")
    ROOT = ROOT.parent
os.chdir(ROOT)                                       # all relative paths below start from here
print("Working folder:", Path.cwd())

**What this cell does:** Imports the libraries and prints their versions and the kernel's Python path.

**Why it matters:** Results are only comparable with the reference run if the library versions are the same. The reference versions are stored in `reference/metrics.json`, and section 12 compares them.

**What to look for:** Python 3.12.12, numpy 2.5.4, pandas 3.1.0, scikit-learn 1.9.1 (the reference versions), and a kernel path ending in `.venv/bin/python`.

In [ ]:
import hashlib, json, platform, sys, zipfile      # standard library: hashing, JSON, versions, unzip
import inspect, io, time                          # standard library: demos and timing

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib.pyplot as plt                   # calibration plot (saved to outputs/ only if SAVE_OUTPUTS)
from IPython.display import display

print("Python      ", platform.python_version())
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("Kernel      ", sys.executable)

**What this cell does:** Holds **every setting** of the baseline in one place, including `N_BOOT` (the number of bootstrap resamples) and `SAVE_OUTPUTS` (whether to write the result files). This is the only cell where settings are defined.

**Why it matters:** One settings cell makes experiments safe and traceable: every change is visible here, and section 12 reports which reference results it changed. The comment next to each setting says what changing it would do.

**What to look for:** Nothing is printed. With these default values, section 12 must print `MATCHES THE REFERENCE MODEL`.

In [ ]:
# ==============================================================================================================
#  SETTINGS: change things here, then rerun from the section named in the section 13 table (or Restart + Run All)
# ==============================================================================================================
DATA_PATH = "data/raw/sample_2015.zip"   # zip (or folder) with sample_orig_*.txt and sample_perf_*.txt. Another
                                         #   vintage means different loans, so nothing would match the reference.
SEED = 20261022                          # seed for the loan bootstrap (also LogisticRegression's random_state, which
                                         #   the lbfgs solver does not use). Changing it should change only the 95% CIs.
TRAIN_SNAPSHOTS = [201512, 201612, 201712, 201812]   # December prediction dates used to FIT the model. Their
                                                     #   12-month windows end by Dec 2019, before the pandemic.
TEST_SNAPSHOTS = [202112, 202212, 202312, 202412]    # dates used to TEST (windows end by Dec 2025; data end 2026-03).
                                                     #   202512 would need data to 2026-12: its non-events could not
                                                     #   be observed, so its labels would be biased.
STRESS_SNAPSHOTS = [201912, 202012]                  # pandemic windows: reported only, never used to fit.
HORIZON = 12                             # outcome window in months (t+1 .. t+HORIZON). Changes every label, so label
                                         #   counts, set sizes, metrics and coefficients all change.
TRAIN_LOAN_SHARE = 70                    # % of loans (hash buckets 0-99) eligible for training; the rest are eligible
                                         #   for test/stress. Changes set sizes and every metric.
C = 1.0                                  # inverse strength of the L2 penalty. Smaller C = stronger shrinkage of the
                                         #   coefficients toward 0. Changes coefficients and metrics, not label counts.
N_BOOT = 1000                            # bootstrap resamples for the 95% CIs. 100 gives a quick run, but the
                                         #   reference used 1000, so any other value changes the CIs.
SAVE_OUTPUTS = False                     # True = at the end of section 12, also write the five result files
                                         #   (metrics.json, label_counts.csv, coefficients.csv, calibration.csv/.png)
                                         #   to outputs/ (gitignored). Changes no result; reference/ is never written.

## 2. Loading the raw files

`sample_2015.zip` contains two **pipe-delimited** text files (`|` between fields):

* `sample_orig_2015.txt`: one row per loan, 31 fields describing the loan **at origination** (credit score,
  loan-to-value, purpose, ...).
* `sample_perf_2015.txt`: one row per loan **per month**, 35 fields (delinquency status, balance, payoff codes, ...).

The General User Guide (July 2026, p. 4) says: *"The origination and monthly performance data files do not contain a
header row. A header and the file layout are available on the SFLLD webpage."* So we supply the column names
ourselves, in the order of the July 2026 file layout (`data/raw/file_layout_july_2026.xlsx`). If the order were
wrong, every column would silently get the wrong name. The count check below and the value checks in section 3
(for example, FICO within 300–850) guard against that.

**What this cell does:** Defines the 31 origination and 35 performance column names, in the order of the July 2026 file layout, and checks the counts.

**Why it matters:** The files have no header row, so these lists are the only thing that tells pandas which field is which.

**What to look for:** No error. The two numbers printed are 31 and 35.

In [ ]:
ORIG_COLS = ["fico", "first_payment_date", "first_time_homebuyer", "maturity_date", "msa", "mi_pct",
             "num_units", "occupancy_status", "orig_cltv", "orig_dti", "orig_upb", "orig_ltv",
             "orig_interest_rate", "channel", "ppm_flag", "amortization_type", "property_state",
             "property_type", "postal_code", "loan_id", "loan_purpose", "orig_loan_term",
             "num_borrowers", "seller_name", "super_conforming_flag", "pre_harp_loan_id",
             "special_eligibility_program", "harp_indicator", "property_valuation_method",
             "io_indicator", "vantagescore4"]
PERF_COLS = ["loan_id", "period", "current_upb", "delinq_status", "loan_age", "rem_months_to_maturity",
             "defect_settlement_date", "modification_flag", "zero_balance_code", "zero_balance_date",
             "current_interest_rate", "current_non_int_upb", "ddlpi", "mi_recoveries",
             "net_sales_proceeds", "non_mi_recoveries", "total_expenses", "legal_costs",
             "maint_pres_costs", "taxes_insurance", "misc_expenses", "actual_loss", "cum_mod_costs",
             "step_mod_flag", "payment_deferral_flag", "eltv", "zb_removal_upb",
             "delinq_accrued_interest", "delinq_due_to_disaster", "borrower_assistance_plan",
             "current_period_mod_costs", "current_int_upb", "mi_cancellation_flag", "servicer_name",
             "bankruptcy_cramdown_costs"]
assert len(ORIG_COLS) == 31 and len(PERF_COLS) == 35   # the documented field counts
print(len(ORIG_COLS), len(PERF_COLS))

**Why read everything as text first?** The files use codes, not just numbers: delinquency status `00`, `01`, `RA`;
zero-balance code `01`; FICO `9999` = "Not Available". If pandas guesses the types, `00` becomes the number 0, `01`
becomes 1 (so a later test `== "01"` silently finds nothing), and `9999` looks like a real (impossible) credit score.
By default pandas also turns strings such as `NA` or `null` into missing values. `load()` therefore reads every
column as text (`dtype=str`, `keep_default_na=False`, `na_values=[""]`): only truly empty fields become missing, and
each code is converted deliberately, in sections 3 and 4.

**What this cell does:** Parses one made-up record twice: with pandas' default type guessing, and with the text settings that `load()` uses.

**Why it matters:** It shows on a harmless synthetic line exactly what these reading options protect against.

**What to look for:** In the default column, `00` → 0, `01` → 1 and `NA` → missing (NaN). In the text column every code is kept exactly as written.

In [ ]:
toy_line = "X01|201712|00|01|9999|NA\n"    # made-up record: loan, period, status, zero-balance code, FICO code, text 'NA'
default = pd.read_csv(io.StringIO(toy_line), sep="|", header=None)                  # pandas guesses the types
as_text = pd.read_csv(io.StringIO(toy_line), sep="|", header=None, dtype=str,
                      keep_default_na=False, na_values=[""])                          # load()'s options
display(pd.DataFrame({"default parsing": default.iloc[0], "as text (load)": as_text.iloc[0]}))

**What this cell does:** Defines `ym_index()`. It turns a `YYYYMM` month into a simple month counter.

**Why it matters:** `load()` (two cells below) uses it to create the month index column `t`. Section 4 explains why it is needed. The leakage test also runs this cell (it carries the cell tag `leakage-test-source`; see section 11).

**What to look for:** Nothing is printed.

In [ ]:
def ym_index(s):
    """YYYYMM -> months since year 0 (so t+12 is simple integer arithmetic)."""
    s = pd.Series(s).astype(int)
    return (s // 100) * 12 + (s % 100) - 1

**What this cell does:** Defines `open_files()`: unzips the sample into `sflld_extracted/` and finds the two text files.

**Why it matters:** It accepts either the zip or an already-unzipped folder, so `DATA_PATH` can point at either.

**What to look for:** Nothing is printed.

In [ ]:
def open_files(data):
    data = Path(data)
    if data.suffix == ".zip":                                   # unzip into ./sflld_extracted (gitignored)
        out = Path("sflld_extracted"); out.mkdir(exist_ok=True)
        with zipfile.ZipFile(data) as z:
            z.extractall(out)
        data = out
    orig = next(data.rglob("sample_orig_*.txt")); perf = next(data.rglob("sample_perf_*.txt"))   # find both files
    return orig, perf

**What this cell does:** Defines `load()`. It reads both files as text, runs three structural checks, and adds the derived performance columns `t`, `dq` and `sev`, which are explained in section 4.

**Why it matters:** Every later step relies on three facts: one row per loan, every monthly record belongs to a known loan, and at most one record per loan per month. If any of them failed, the snapshot logic would double-count or lose loans.

**What to look for:** Nothing is printed yet. The function runs in the next cell.

In [ ]:
def load(data):
    of, pf = open_files(data)
    orig = pd.read_csv(of, sep="|", header=None, names=ORIG_COLS, dtype=str,     # every field as text
                       keep_default_na=False, na_values=[""])
    usecols = ["loan_id", "period", "current_upb", "delinq_status", "loan_age", "modification_flag",
               "zero_balance_code", "delinq_due_to_disaster", "borrower_assistance_plan"]   # 9 of the 35 fields
    perf = pd.read_csv(pf, sep="|", header=None, names=PERF_COLS, dtype=str, usecols=usecols,
                       keep_default_na=False, na_values=[""])
    # structural checks against the documentation
    assert orig.loan_id.is_unique, "duplicate loan IDs in origination file"
    assert set(perf.loan_id) <= set(orig.loan_id), "performance loans missing from origination"
    assert not perf.duplicated(["loan_id", "period"]).any(), "duplicate loan-month records"
    perf["t"] = ym_index(perf.period).values                    # month index (section 4)
    dq = pd.to_numeric(perf.delinq_status, errors="coerce")     # 'RA' -> NaN here
    perf["dq"] = dq                                             # status as a number: 0, 1, 2, 3, ...
    perf["sev"] = (dq >= 3) | (perf.delinq_status == "RA")       # 90+ or REO acquisition
    perf["current_upb"] = pd.to_numeric(perf.current_upb)       # text -> number
    perf["loan_age"] = pd.to_numeric(perf.loan_age)
    print(f"Loaded {len(orig):,} loans, {len(perf):,} monthly records, "
          f"periods {perf.period.min()}-{perf.period.max()}")
    return orig, perf

**What this cell does:** Sets the global random seed and loads the data.

**Why it matters:** This is the starting point for everything else.

**What to look for:** `Loaded 50,000 loans, 3,467,166 monthly records, periods 201501-202603`. It takes a few seconds.

In [ ]:
np.random.seed(SEED)            # as in the original script's main(); kept so every run starts the same way
orig, perf = load(DATA_PATH)

**What this cell does:** Prints the numbers behind the three structural checks in `load()`.

**Why it matters:** An `assert` only says pass or fail. Seeing the counts confirms that the data match the documentation and the facts recorded in `CLAUDE.md`.

**What to look for:** 50,000 rows and 50,000 unique loan IDs; 3,467,166 monthly records; 0 unknown loans; 0 duplicate loan-months; months from 201501 to 202603.

In [ ]:
print(f"Origination: {len(orig):,} rows, {orig.loan_id.nunique():,} unique loan IDs, {orig.shape[1]} columns")
print(f"Performance: {len(perf):,} monthly records for {perf.loan_id.nunique():,} loans, "
      f"{perf.shape[1]} columns (9 read from the file + t, dq, sev)")
print("Performance loans missing from the origination file:", len(set(perf.loan_id) - set(orig.loan_id)))
print("Duplicate loan-months:", int(perf.duplicated(["loan_id", "period"]).sum()))
print(f"Months: {perf.period.min()} to {perf.period.max()} ({perf.period.nunique()} distinct months)")

**What this cell does:** Shows the first 5 rows of each table.

**Why it matters:** It shows what the raw data look like: every origination field is still text, and the performance table has one row per loan-month.

**What to look for:** Codes such as `00` and `9999` are still text. In the performance table, consecutive months for the same loan, with `t` increasing by 1.

In [ ]:
display(orig.head(5))          # 5 real loans, all 31 origination fields, still as text
display(perf.head(5))          # their first monthly records, with the derived t, dq and sev

## 3. Preprocessing the origination data

Several numeric fields use a special value instead of an empty field when a value is "Not Available". These values
are **codes, not numbers**: a DTI of 999% is not a very indebted borrower; the ratio is simply not disclosed. Used as
a number, 999 would distort the model (benchmark pair 1 plants exactly this defect). The baseline lists the documented
codes in `SENTINELS` and turns them into missing values (NaN) before converting the fields to numbers.

**What this cell does:** Defines `SENTINELS`: for each numeric field, the code the guide documents as "Not Available".

**Why it matters:** These are the only values the baseline turns into missing values, so each one must be backed by the documentation (next cell).

**What to look for:** Nothing is printed.

In [ ]:
# Documented "Not Available" codes (General User Guide, July 2026) -> converted to NaN
SENTINELS = {"fico": "9999", "mi_pct": "999", "num_units": "99", "orig_cltv": "999",
             "orig_dti": "999", "orig_ltv": "999", "num_borrowers": "99"}

**What this cell does:** Builds the table of documented codes (field, code, meaning, source) and counts each code **before** and **after** the conversion to missing.

**Why it matters:** The answer key of the benchmark must be defensible from the documentation. This table links every conversion to a page of the General User Guide and shows how many loans it affects.

**What to look for:** DTI `999`: 4,020 before, 0 after, 4,020 missing. LTV and CLTV `999`: 1 loan each. FICO, MI, units and borrowers: the codes are documented but do not occur in this sample (0). Every "after" count is 0.

In [ ]:
GUIDE = {   # meaning in the General User Guide, July 2026 (page numbers as printed in the guide)
    "fico":          ("Not Available (scores < 300 or > 850 are also disclosed as Not Available)", "4"),
    "mi_pct":        ("Not Available (MI < 0% or > 55% is also disclosed as Not Available; 0 = no MI)", "4-5"),
    "num_units":     ("Not Available (values < 1 or > 4 are also disclosed as Not Available)", "5"),
    "orig_cltv":     ("Not Available (outside the documented range, or CLTV < LTV)", "5"),
    "orig_dti":      ("Not Available (ratios > 65%; ALL HARP loans are disclosed as Not Available)", "5"),
    "orig_ltv":      ("Not Available (outside the documented range)", "6"),
    "num_borrowers": ("Not Available (values < 1 or > 10 are also disclosed as Not Available)", "8"),
}
after = orig.copy()
for c, code in SENTINELS.items():                   # the same loop as the first lines of origination_features()
    after.loc[after[c] == code, c] = np.nan
na_table = pd.DataFrame([{"field": c, "code": code, "meaning": GUIDE[c][0],
                          "source": f"General User Guide, July 2026, p. {GUIDE[c][1]}",
                          "code count before": int((orig[c] == code).sum()),
                          "code count after": int((after[c] == code).sum()),
                          "missing after": int(after[c].isna().sum())} for c, code in SENTINELS.items()])
with pd.option_context("display.max_colwidth", None):
    display(na_table)

**What this cell does:** Cross-tabulates DTI = 999 against the HARP indicator (Freddie Mac's Relief Refinance program).

**Why it matters:** The guide says *"All HARP loans in the dataset will be disclosed as Not Available"* for DTI. So a missing DTI mostly means "this is a HARP refinance", not "the borrower hid their income". That is why the baseline keeps the fact that DTI was missing as its own feature, `dti_missing`.

**What to look for:** 4,017 of the 4,020 loans with DTI 999 are HARP loans, and every HARP loan has DTI 999. The other 3 are non-HARP loans; the guide's other documented reason for 999 is a ratio above 65%.

In [ ]:
dti_999 = orig.orig_dti.eq("999").rename("DTI is 999")
display(pd.crosstab(dti_999, orig.harp_indicator.rename("HARP indicator"), margins=True))
print(f"{int((dti_999 & orig.harp_indicator.eq('Y')).sum()):,} of {int(dti_999.sum()):,} loans with DTI 999 are HARP loans")

**What this cell does:** Defines `origination_features()`: codes → NaN, text → numbers, five derived features, and the 16 columns that are kept.

**Why it matters:** These are the loan-level features the model sees. Origination data never change after the loan is made, so they cannot leak future information.

**What to look for:** Nothing is printed.

In [ ]:
def origination_features(orig):
    o = orig.copy()
    for c, code in SENTINELS.items():                  # documented "Not Available" codes -> NaN
        o.loc[o[c] == code, c] = np.nan
    num = ["fico", "mi_pct", "num_units", "orig_cltv", "orig_dti", "orig_ltv",
           "orig_interest_rate", "orig_upb", "orig_loan_term", "num_borrowers"]
    for c in num:                                      # text -> numbers (after the codes are removed)
        o[c] = pd.to_numeric(o[c])
    o["dti_missing"] = o.orig_dti.isna().astype(int)   # almost all are HARP loans (guide: HARP DTI = N/A)
    o["log_orig_upb"] = np.log(o.orig_upb)             # log of the original loan amount
    o["term_360"] = (o.orig_loan_term == 360).astype(int)      # 1 = 30-year loan
    o["multi_borrower"] = (o.num_borrowers >= 2).astype(int)   # 1 = two or more borrowers
    o["mi_flag"] = (o.mi_pct > 0).astype(int)                  # 1 = has mortgage insurance
    keep = ["loan_id", "fico", "orig_ltv", "orig_dti", "dti_missing", "orig_interest_rate",
            "log_orig_upb", "term_360", "multi_borrower", "mi_flag", "num_units",
            "loan_purpose", "occupancy_status", "first_time_homebuyer", "property_type", "channel"]
    return o[keep]                                     # one row per loan

**What this cell does:** Runs `origination_features()` and summarizes the numeric columns.

**Why it matters:** `describe()` is a quick plausibility check: impossible values here would point to a wrong column order or a missed code.

**What to look for:** Shape (50000, 16). FICO between 300 and 850; `orig_dti` has a count of 45,980 (4,020 missing) and a maximum of at most 65; `orig_ltv` has one missing value (count 49,999). LTV values above 105% are possible: the guide (p. 6) allows HARP loans 1–998%.

In [ ]:
ofeat = origination_features(orig)
print(ofeat.shape)
display(ofeat.describe().T.round(2))

**The five derived features, one sentence each.**

* `dti_missing` = 1 when DTI was the 999 code (now NaN). The model can then treat "DTI not disclosed" (almost
  always a HARP refinance) separately from the DTI value, which section 8 fills with the training median.
* `log_orig_upb` = natural log of the original loan amount (UPB = unpaid principal balance). Loan sizes are
  right-skewed; on the log scale, doubling the loan size is always the same step.
* `term_360` = 1 for a 360-month (30-year) loan. Shorter terms, mostly 15-year loans, repay principal faster.
* `multi_borrower` = 1 when two or more borrowers are obligated on the loan. A missing count (code 99) would give 0,
  but no loan in this sample has that code.
* `mi_flag` = 1 when the loan has primary mortgage insurance (`mi_pct > 0`). A missing MI value (code 999) would give
  0, but no loan in this sample has that code.

**What this cell does:** Shows the raw text and the derived value side by side for 5 loans: the first 4 loans with a reported DTI, plus the first loan with DTI 999.

**Why it matters:** It shows on real rows that each derived feature does what its one-sentence description says.

**What to look for:** In the DTI-999 row, `dti_missing` = 1. `term_360` is 1 exactly when the term is `360`, `multi_borrower` is 1 when there are 2 borrowers, and `mi_flag` is 1 when `mi_pct` is above 0.

In [ ]:
rows = list(orig[orig.orig_dti.ne("999")].index[:4]) + [orig.orig_dti.eq("999").idxmax()]   # 4 + 1 loan with DTI 999
before_after = pd.DataFrame({
    "orig_dti (raw)": orig.loc[rows, "orig_dti"], "dti_missing": ofeat.loc[rows, "dti_missing"],
    "orig_upb (raw)": orig.loc[rows, "orig_upb"], "log_orig_upb": ofeat.loc[rows, "log_orig_upb"].round(3),
    "orig_loan_term (raw)": orig.loc[rows, "orig_loan_term"], "term_360": ofeat.loc[rows, "term_360"],
    "num_borrowers (raw)": orig.loc[rows, "num_borrowers"], "multi_borrower": ofeat.loc[rows, "multi_borrower"],
    "mi_pct (raw)": orig.loc[rows, "mi_pct"], "mi_flag": ofeat.loc[rows, "mi_flag"]})
display(before_after)

**What this cell does:** Counts the 0s and 1s of the four 0/1 features.

**Why it matters:** A flag that is almost always 0, or always 1, carries little information. It is worth knowing how balanced each flag is.

**What to look for:** `dti_missing` = 1 for 4,020 loans; `term_360` = 1 for 36,501; `multi_borrower` = 1 for 25,540; `mi_flag` = 1 for 11,729.

In [ ]:
flags = ["dti_missing", "term_360", "multi_borrower", "mi_flag"]
display(pd.DataFrame({f: ofeat[f].value_counts().sort_index() for f in flags}).rename_axis("value"))

**A note on the categorical "9"/"99" codes.** The categorical fields also have "Not Available" codes: `9` for loan
purpose, occupancy status, first-time homebuyer and channel, and `99` for property type (guide pp. 4–8). The baseline
does **not** convert them: they stay as their own category. This matters little here, because only **8 loans** in the
whole sample have such a code, all of them in `first_time_homebuyer`. Because the one-hot encoder in section 8 groups
levels seen fewer than 20 times in training into one "infrequent" column, these loans end up in
`first_time_homebuyer_infrequent_sklearn`. Exercise 1 in section 13 maps them to an explicit "unknown" category.

**What this cell does:** Counts the values of the five categorical origination fields used by the model, and the loans with a `9`/`99` code in any of them.

**Why it matters:** It supports the note above with numbers, rather than leaving it as a claim.

**What to look for:** Only `first_time_homebuyer` has a `9` (8 loans). The last line prints 8.

In [ ]:
cat_cols = ["loan_purpose", "occupancy_status", "first_time_homebuyer", "property_type", "channel"]
for c in cat_cols:
    print(f"{c:22s}", orig[c].value_counts().to_dict())
print("\nLoans with a '9' or '99' code in any of these fields:", int(orig[cat_cols].isin(["9", "99"]).any(axis=1).sum()))

## 4. Preprocessing the monthly performance data

**The month index $t$.** Periods are stored as `YYYYMM` numbers. They sort correctly, but you cannot subtract them:
$201601 - 201512 = 89$, although the two months are 1 month apart. `ym_index()` therefore converts every period into a
simple month counter:

$$ t \;=\; 12 \cdot \left\lfloor \frac{\text{YYYYMM}}{100} \right\rfloor \;+\; (\text{YYYYMM} \bmod 100) \;-\; 1 $$

For example, $201512 \to 12 \cdot 2015 + 12 - 1 = 24191$ and $201601 \to 12 \cdot 2016 + 1 - 1 = 24192$:
exactly one apart. "12 months after $t$" is then simply $t + 12$.

**What this cell does:** Applies `ym_index()` to four consecutive months and compares the steps with naive `YYYYMM` subtraction.

**Why it matters:** The outcome window ($t+1 \dots t+12$) and the lookback ($t-11 \dots t$) are computed with this index. An error here would shift every window.

**What to look for:** The `YYYYMM` steps are 1, 89, 1; the `t` steps are always 1. `t` for 201712 plus 12 equals `t` for 201812 (`True`).

In [ ]:
months = pd.Series([201511, 201512, 201601, 201602])
demo = pd.DataFrame({"YYYYMM": months, "t = ym_index(YYYYMM)": ym_index(months).values})
demo["step in YYYYMM"] = demo["YYYYMM"].diff()               # naive subtraction of the codes
demo["step in t"] = demo["t = ym_index(YYYYMM)"].diff()      # the true number of months
display(demo)
t_dec17 = int(ym_index([201712])[0])
print("t(201712) =", t_dec17, "| t(201712) + 12 == t(201812):", t_dec17 + 12 == int(ym_index([201812])[0]))

**Delinquency status codes** (guide p. 10). The status is based on the due date of the last paid installment:

| code | meaning | `dq` (number) | `sev` (serious) |
|---|---|---|---|
| `00` | current, or less than 30 days late | 0 | False |
| `01` | 30–59 days late | 1 | False |
| `02` | 60–89 days late | 2 | False |
| `03`, `04`, … | 90–119 days, 120–149 days, … (capped at 99) | 3, 4, … | **True** |
| `RA` | REO acquisition: the lender has taken the property | NaN (not a number) | **True** |
| `XX` | not available | NaN | False |

`load()` adds two columns: `dq`, the status as a number (`RA` → NaN), and `sev` = (`dq` ≥ 3) or (status = `RA`),
meaning "90+ days late or REO". `sev` drives both the at-risk rule (a loan with `sev` at $t$ is excluded) and the
event definition (any `sev` month in $t+1 \dots t+12$). It looks only at each month's value, not at the path: a loan
can jump from `00` to `03` in one month (a known false-alarm trap listed in `CLAUDE.md`). The guide also notes that,
for periods up to September 2025, `00` can also mean "cycle before the first payment date".

**What this cell does:** Counts the monthly records for each delinquency status code, with the `dq` and `sev` values `load()` gave them.

**Why it matters:** It checks the code-to-number mapping on the real data and shows how rare serious delinquency is.

**What to look for:** `00` 3,415,433 records; `01` 23,522; `02` 6,686; `03` 3,756; `RA` 567 with `dq` NaN and `sev` True. Every code from `03` up has `sev` True. The only non-numeric code is `RA`; there are no `XX` records. 21,525 records have `sev` True.

In [ ]:
by_code = perf.groupby("delinq_status").agg(records=("t", "size"), dq=("dq", "first"), sev=("sev", "first"))
shown = ["00", "01", "02", "03", "04", "05", "RA"]
display(by_code.loc[shown])
rest = by_code.drop(shown)
print(f"Other codes: {len(rest)} values from {rest.index.min()} to {rest.index.max()}, "
      f"{rest.records.sum():,} records, all with sev = True: {bool(rest.sev.all())}")
print("Non-numeric codes:", sorted(by_code.index[by_code.dq.isna()]),
      "| 'XX' records:", int((perf.delinq_status == "XX").sum()))
print(f"Records with sev = True: {int(perf.sev.sum()):,} of {len(perf):,}")

**What this cell does:** Counts the zero-balance codes: the reason a loan's balance went to zero and it left the data.

**Why it matters:** These codes decide the censored labels in section 5. Code `01` (prepaid) becomes `censored_prepay`; any other code becomes `censored_other_termination`, if the loan leaves before $t+12$ without a 90+ month.

**What to look for:** `01` (prepaid) dominates with 38,212 loans; the others are rare (`16`: 108, `09`: 55, `02`: 45, `96`: 35, `15`: 18, `03`: 18). Almost all records have no code, because the loan is still active that month.

In [ ]:
ZB_MEANING = {"01": "Prepaid or matured (voluntary payoff)", "02": "Third party sale", "03": "Short sale or charge off",
              "09": "REO disposition", "15": "Whole loan sale", "16": "Reperforming loan securitization",
              "96": "Confirmed underwriting/major servicing defect prior to credit event"}   # guide p. 11
zb = perf.zero_balance_code.value_counts().rename("records").to_frame()
zb["meaning (guide p. 11)"] = zb.index.map(ZB_MEANING)
zb["label if it happens before t+12 with no 90+ month"] = np.where(zb.index == "01", "censored_prepay",
                                                                    "censored_other_termination")
display(zb)
print(f"Records without a zero-balance code (loan still active): {int(perf.zero_balance_code.isna().sum()):,}")

## 5. A toy example to understand labels and features (synthetic data)

`build_snapshot(perf, T)` turns the monthly panel into **one row per loan** for one prediction month $T$ (a
December). For $T$ = December 2017:

```
                features: months <= t only               label: months t+1 .. t+12 (HORIZON)
   2017-01 ............................ 2017-12 = t | 2018-01 ............................ 2018-12 | 2019-01 ...
   |<------- 12-month lookback: t-11 .. t ------->| |<----------- outcome window ----------------->|
```

The rules (`CLAUDE.md`, "Outcome specification"):

* **At risk at $t$** (gets a row): the loan has a record at $t$, no zero-balance code, and status `00`, `01` or `02`
  at $t$ (`sev` is False). A loan that left before $t$ has no record at $t$, because a terminated loan gets no
  further monthly records (guide p. 16). So checking the code at month $t$ covers "at or before $t$".
* **Event** (`event`, y = 1): any month in $t+1 \dots t+12$ with `sev` True (status `03` or worse, or `RA`).
* **Non-event** (`non_event`, y = 0): no event, and the loan still has a record at month $t+12$.
* **Censored** (counted, but not used for fitting): no event, and the loan leaves before $t+12$.
  `censored_prepay` = zero-balance code `01`; `censored_other_termination` = any other code;
  `censored_end_of_data` = the records stop without a code (the window runs past the end of the data).

The order matters: the event check comes first. A loan that goes 90+ and then pays off inside the window is still an
**event** (another false-alarm trap in `CLAUDE.md`: loans can prepay while 90+).

**What this cell does:** Defines `build_snapshot()`. It has three blocks: the at-risk population at $t$; features computed **only** from `hist` (months ≤ $t$); and the label from the window $t+1 \dots t+$`HORIZON`.

**Why it matters:** This is the only place where time enters the pipeline. A mistake here means either leakage (features that see the future) or a wrong outcome. Benchmark pairs 3 (horizon) and 4 (leakage) plant their defects exactly here. The leakage test runs this cell directly (it carries the cell tag `leakage-test-source`; see section 11).

**What to look for:** Nothing is printed. Notice that every feature line uses `hist`, `now`, `h12`, `ever` or `hh`, all of which are filtered to months ≤ $t$. Only the label block uses `perf` directly.

In [ ]:
def build_snapshot(perf, T):
    t = int(ym_index([T])[0])                                  # prediction month as a month index
    hist = perf[perf.t <= t]                                   # ONLY information at or before t
    now = hist[hist.t == t]                                    # each loan's record AT month t
    now = now[now.zero_balance_code.isna() & ~now.sev]         # at-risk population
    ids = now.loan_id

    h12 = hist[(hist.t > t - 12) & hist.loan_id.isin(ids)]     # lookback window t-11 .. t
    g12 = h12.groupby("loan_id")
    ever = hist[hist.loan_id.isin(ids)].groupby("loan_id")     # the whole history up to t
    feat = now.set_index("loan_id")[["delinq_status", "loan_age", "current_upb",
                                     "delinq_due_to_disaster", "borrower_assistance_plan"]].copy()
    feat["status_t"] = feat.pop("delinq_status")               # '00', '01', '02'
    feat["max_dq_12m"] = g12.dq.max().clip(upper=3)            # 0..3 (3 = was 90+ in last 12m)
    feat["n_dq_months_12m"] = (h12.dq >= 1).groupby(h12.loan_id).sum()        # months 30+ days late in t-11..t
    feat["ever_90_before_t"] = ever.sev.any().astype(int)      # ever 90+ or REO at or before t
    hh = hist[hist.loan_id.isin(ids)]
    feat["ever_modified"] = hh.modification_flag.notna().groupby(hh.loan_id).any().astype(int)   # any Y/P flag
    feat["disaster_flag_t"] = feat.pop("delinq_due_to_disaster").eq("Y").astype(int)  # disaster flag at t
    feat["assist_plan_t"] = feat.pop("borrower_assistance_plan").fillna("none")       # F/R/T at t, else 'none'
    # Leakage is tested by tests/test_baseline_no_leakage.py (future-invariance + mutation check).

    # ---- label (future window, never used as a feature)
    win = perf[(perf.t > t) & (perf.t <= t + HORIZON) & perf.loan_id.isin(ids)]      # months t+1 .. t+HORIZON
    event = win.groupby("loan_id").sev.any().reindex(ids, fill_value=False)          # any 90+/REO month?
    last_t = perf[perf.loan_id.isin(ids)].groupby("loan_id").t.max().reindex(ids)     # last month observed
    zb = (perf[perf.loan_id.isin(ids) & perf.zero_balance_code.notna()]
          .set_index("loan_id").zero_balance_code.reindex(ids))                        # zero-balance code, if any
    label = np.where(event.values, "event",
             np.where(last_t.values >= t + HORIZON, "non_event",
              np.where(zb.values == "01", "censored_prepay",
               np.where(pd.notna(zb.values), "censored_other_termination", "censored_end_of_data"))))
    feat["label"] = label
    feat["snapshot"] = T
    return feat.reset_index()

**The five made-up loans.** IDs `X01`–`X05` (deliberately not the Freddie Mac `F15Q…` pattern). Prediction month
$T$ = December 2017, so the lookback is 2017-01 … 2017-12 and the outcome window is 2018-01 … 2018-12.

| loan | story | expected |
|---|---|---|
| `X01` | current the whole time | at risk → `non_event` |
| `X02` | 30 days late in Sep 2017, current at $t$, then 30 → 60 → 90 days late in Mar–May 2018 | at risk → `event` |
| `X03` | 30 days late **at** $t$, current again in Jan 2018, prepays in Apr 2018 | at risk → `censored_prepay` |
| `X04` | 30 → 60 → 90 days late in Oct–Dec 2017: already 90+ **at** $t$ | **not at risk** → no row |
| `X05` | current through Dec 2018; late only from Jan 2019 ($t+13$) | at risk → `non_event` |

**What this cell does:** Writes down the five made-up loans as one line of monthly status codes each, from 201701 to 201906.

**Why it matters:** With five loans and 30 months, every label and feature can be checked by hand, which is impossible on 3.5 million real records. Writing each loan as a timeline makes its story visible at a glance.

**What to look for:** Read each line as a timeline: the column under `D` of 2017 is month $t$. `X04` is `03` there; `X03` is `01` there, then `--` (no records) after its payoff month 201804.

In [ ]:
T_TOY = 201712                                                     # toy prediction month t (December 2017)
TOY_PERIODS = [f"{y}{m:02d}" for y in (2017, 2018, 2019) for m in range(1, 13)][:30]   # 201701 .. 201906

# Monthly delinquency status of each made-up loan, one two-character code per month; "--" = no record.
TOY_STATUS = {
    #       2017 (t = December 2017)              |  2018 = outcome window t+1..t+12      |  2019
    #       J  F  M  A  M  J  J  A  S  O  N  D   |  J  F  M  A  M  J  J  A  S  O  N  D   |  J  F  M  A  M  J
    "X01": "00 00 00 00 00 00 00 00 00 00 00 00  |  00 00 00 00 00 00 00 00 00 00 00 00  |  00 00 00 00 00 00",   # stays current
    "X02": "00 00 00 00 00 00 00 00 01 00 00 00  |  00 00 01 02 03 04 05 06 07 08 09 10  |  11 12 13 14 15 16",   # 90+ within 12 months
    "X03": "00 00 00 00 00 00 00 00 00 00 00 01  |  00 00 00 00 -- -- -- -- -- -- -- --  |  -- -- -- -- -- --",   # prepays in 201804
    "X04": "00 00 00 00 00 00 00 00 00 01 02 03  |  04 05 06 07 08 09 10 11 12 13 14 15  |  16 17 18 19 20 21",   # already 90+ at t
    "X05": "00 00 00 00 00 00 00 00 00 00 00 00  |  00 00 00 00 00 00 00 00 00 00 00 00  |  01 02 03 04 05 06",   # late only after t+12
}

**What this cell does:** Turns the timelines into a monthly panel with exactly the columns and types that `load()` produces, then adds `t`, `dq` and `sev` with the same three lines as `load()`.

**Why it matters:** `build_snapshot()` must receive the same kind of table as in the real run, so the toy result means something for the real data. Balances and loan ages are made up; the other fields are left empty (no modification, disaster or assistance flags).

**What to look for:** 136 rows: 30 months for four loans and 16 for `X03`. In the `X03` rows, the last record (201804) has zero-balance code `01`.

In [ ]:
TOY_AGE_JAN2017 = {"X01": 25, "X02": 7, "X03": 13, "X04": 19, "X05": 31}                       # loan age in 201701
TOY_UPB = {"X01": 210000.0, "X02": 145000.0, "X03": 320000.0, "X04": 98000.0, "X05": 175000.0}  # flat, made up

toy_rows = []
for loan, line in TOY_STATUS.items():
    codes = [s for s in line.split() if s != "|"]                # 30 codes, one per month
    assert len(codes) == len(TOY_PERIODS)
    last = max(i for i, s in enumerate(codes) if s != "--")      # position of the loan's last record
    for i, (period, status) in enumerate(zip(TOY_PERIODS, codes)):
        if status == "--":                                       # no record: the loan has left the data
            continue
        paid_off = codes[-1] == "--" and i == last               # a loan that leaves early prepaid on its last record
        toy_rows.append({"loan_id": loan, "period": period, "current_upb": TOY_UPB[loan], "delinq_status": status,
                         "loan_age": TOY_AGE_JAN2017[loan] + i, "modification_flag": None,
                         "zero_balance_code": "01" if paid_off else None,
                         "delinq_due_to_disaster": None, "borrower_assistance_plan": None})
text_cols = ["loan_id", "period", "delinq_status", "modification_flag", "zero_balance_code",
             "delinq_due_to_disaster", "borrower_assistance_plan"]
toy = pd.DataFrame(toy_rows).astype({c: "str" for c in text_cols})   # text columns, as load() reads them
toy["t"] = ym_index(toy.period).values                       # the same three derived columns as in load()
dq = pd.to_numeric(toy.delinq_status, errors="coerce")
toy["dq"] = dq
toy["sev"] = (dq >= 3) | (toy.delinq_status == "RA")
print("Toy panel:", toy.shape, "| records per loan:", toy.groupby("loan_id").size().to_dict())

**What this cell does:** Shows the `X03` records from 201710 to 201805.

**Why it matters:** This is what one loan looks like in the long, one-row-per-month format that `build_snapshot()` receives.

**What to look for:** 201712 (= $t$) has status `01`; 201804 is the last record and carries zero-balance code `01`; there is no 201805 record.

In [ ]:
display(toy[(toy.loan_id == "X03") & toy.period.between("201710", "201805")])

**What this cell does:** Repeats, outside the function, the first step of `build_snapshot()`: the at-risk test at month $t$.

**Why it matters:** It shows which loans get a row at all, before any feature or label is computed.

**What to look for:** `X04` has `sev` True at $t$ (status `03`), so `at_risk` is False. The other four loans are at risk, including `X03` with status `01`.

In [ ]:
t = int(ym_index([T_TOY])[0])
at_t = toy[toy.t == t][["loan_id", "period", "delinq_status", "zero_balance_code", "sev"]].copy()
at_t["at_risk"] = at_t.zero_balance_code.isna() & ~at_t.sev          # the same rule as in build_snapshot()
display(at_t)

**What this cell does:** For each at-risk loan, collects the ingredients of the label: the window months, whether any of them is 90+, the last month observed, and the zero-balance code.

**Why it matters:** The label is a small decision tree (event? → observed at $t+12$? → which code?). Seeing its inputs makes every label explainable.

**What to look for:** `X02`: event True, first 90+ month 201805. `X03`: only 4 window months, last observed 201804, code `01`. `X01` and `X05`: no 90+ in the window and observed at 201812, so they will be non-events.

In [ ]:
ids = at_t.loc[at_t.at_risk, "loan_id"]
win = toy[(toy.t > t) & (toy.t <= t + HORIZON) & toy.loan_id.isin(ids)]        # months t+1 .. t+HORIZON
end = t + HORIZON
end_period = f"{end // 12}{end % 12 + 1:02d}"                                   # t+HORIZON back to YYYYMM (201812)
ingredients = pd.DataFrame({
    "months in window": win.groupby("loan_id").size(),
    "any 90+ in window (event)": win.groupby("loan_id").sev.any(),
    "first 90+ month": win[win.sev].groupby("loan_id").period.min(),
    "last month observed": toy[toy.loan_id.isin(ids)].groupby("loan_id").period.max(),
    "zero-balance code": toy[toy.zero_balance_code.notna()].set_index("loan_id").zero_balance_code,
}).reindex(ids.values)
ingredients[f"observed at t+{HORIZON} ({end_period})"] = ingredients["last month observed"] >= end_period
display(ingredients)

**What this cell does:** Runs the real `build_snapshot()` on the toy panel.

**Why it matters:** This is the output format of every real snapshot in section 6: one row per at-risk loan, the features, the label and the snapshot date.

**What to look for:** Four rows: `X01`, `X02`, `X03`, `X05` (`X04` is absent). Labels: `non_event`, `event`, `censored_prepay`, `non_event`. The walk-through below explains each value.

In [ ]:
snap_toy = build_snapshot(toy, T_TOY)
display(snap_toy)

### Walking through the toy snapshot, loan by loan

**What the columns mean.**
`loan_age` and `current_upb` are copied from the month-$t$ record (`current_upb` is in the snapshot but is **not** a
model feature; see exercise 2). `status_t` is the status at $t$, always `00`, `01` or `02`, because only at-risk
loans are here. `max_dq_12m` is the worst status in $t-11 \dots t$, capped at 3. `n_dq_months_12m` counts the months
in $t-11 \dots t$ that were at least 30 days late. `ever_90_before_t` = 1 if the loan was ever 90+ or REO at or
before $t$; since loans that are 90+ **at** $t$ are excluded, this can only be a loan that was 90+ earlier and then
cured. `ever_modified` = 1 if any record up to $t$ has a modification flag (`Y` or `P`). `disaster_flag_t` = 1 if the
disaster flag is `Y` at $t$. `assist_plan_t` is the borrower-assistance plan at $t$ (`F` forbearance, `R` repayment,
`T` trial) or `none`. The toy loans have no modification, disaster or assistance records, so those three are 0 /
`none` for all of them; the synthetic loans `T06`–`T08` in `tests/test_baseline_no_leakage.py` exercise them.

**`X01`, stays current.** It has a record at $t$, no zero-balance code, and status `00`, so it is at risk. Every
lookback month is `00`: `max_dq_12m` = 0, `n_dq_months_12m` = 0, `ever_90_before_t` = 0. No window month is 90+, and
the loan still has a record in 201812 ($t+12$), so the label is **`non_event`** (y = 0). `loan_age` = 25 + 11 = 36.

**`X02`, becomes 90+ within 12 months.** Status `00` at $t$, so it is at risk. Its 30-day delinquency in 201709 lies
inside the lookback (201701–201712), so `max_dq_12m` = 1 and `n_dq_months_12m` = 1: the features carry an early
warning sign. In the window it goes `01` (201803), `02` (201804), `03` (201805). The first 90+ month is 201805 =
$t+5$, so the label is **`event`** (y = 1). None of the months after $t$ appears in any feature. `loan_age` = 18.

**`X03`, prepays early.** Status `01` (30 days late) at $t$. It is still at risk: the rule excludes only `03` and
worse. Month $t$ itself is part of the lookback, so `max_dq_12m` = 1 and `n_dq_months_12m` = 1. The loan is current
again in 201801 and pays off in 201804 (zero-balance code `01`), after which its records stop. It had no 90+ month
before leaving, and it was not observed at $t+12$, so the label is **`censored_prepay`**. Nobody knows whether it would
have gone 90+ had it stayed, so this row is counted but not used to fit the model (section 7). `loan_age` = 24.

**`X04`, already 90+ at $t$.** Status `03` at $t$ means `sev` is True, so it is not at risk and gets **no row**.
Asking "will it become 90+?" makes no sense for a loan that already is.

**`X05`, late only after $t+12$.** Current at $t$ and through the whole window 201801–201812, and observed at 201812,
so the label is **`non_event`**. Its 30-day delinquency in 201901 ($t+13$) and its 90+ month 201903 ($t+15$) fall
outside the 12-month window and do not count. With `HORIZON = 15` it would be an event. `loan_age` = 42.

The two labels the toy does not show: `censored_other_termination` (the loan leaves before $t+12$ with a code other
than `01`, for example `09` REO disposition, without a 90+ month first) and `censored_end_of_data` (the records stop
without any code; section 11 produces it by deleting the future).

## 6. Building the real snapshots

The same function now runs on the real panel, once for each December in the settings, in the reference order
(train, stress, test). The origination features from section 3 are then merged in by `loan_id`.

**What this cell does:** Builds every December snapshot, stacks them, and merges in the origination features (`ofeat`, section 3).

**Why it matters:** This is the modelling table. A loan appears once per December in which it is at risk, so the same loan can appear in up to 10 snapshots. `validate="many_to_one"` makes pandas check that each loan has exactly one origination row.

**What to look for:** Shape (273985, 27): 273,985 loan-snapshot rows; 12 columns from `build_snapshot()` + 15 origination features. It takes several seconds.

In [ ]:
snaps = pd.concat([build_snapshot(perf, T) for T in TRAIN_SNAPSHOTS + STRESS_SNAPSHOTS + TEST_SNAPSHOTS])
snaps = snaps.merge(ofeat, on="loan_id", how="left", validate="many_to_one")
print(snaps.shape)
display(snaps.head(5))

**What this cell does:** Counts the labels per snapshot (all loans, before the train/test split). The reference copy is `reference/label_counts.csv` (saved again as `outputs/label_counts.csv` when `SAVE_OUTPUTS = True`).

**Why it matters:** Label counts are the first thing to compare with the reference: if they differ, the outcome rules differ.

**What to look for:** 10 rows (201512 … 202412) and 4 label columns. December 2019 has 1,199 events, far more than any other year. There is no `censored_end_of_data` column: every window ends before the data end.

In [ ]:
counts = snaps.groupby(["snapshot", "label"]).size().unstack(fill_value=0)
display(counts)

**What this cell does:** Compares the label counts with the reference file `reference/label_counts.csv`.

**Why it matters:** This is an early check on the outcome rules, before any model is fitted. Section 12 repeats it in detail.

**What to look for:** `True`. If it prints `False`, section 12 lists exactly which counts differ.

In [ ]:
ref_counts = pd.read_csv("reference/label_counts.csv", index_col="snapshot")
same = (list(counts.index) == list(ref_counts.index) and list(counts.columns) == list(ref_counts.columns)
        and bool((counts.to_numpy() == ref_counts.to_numpy()).all()))
print("Label counts identical to reference/label_counts.csv:", same)

**What this cell does:** Computes, for each snapshot, how many loans are at risk, the event rate among uncensored loans, and the share of censored loans.

**Why it matters:** The event rate is the base rate the model has to learn. Its changes over time explain the stress-test design and the calibration gap in section 9.

**What to look for:** Training years: about 0.24–0.65%. 201912: about 4.9%. Test years: about 0.67–0.90%, higher than in training. The censored share (mostly prepayments) is highest in 201912 and 202012.

In [ ]:
period_of = {**{T: "train" for T in TRAIN_SNAPSHOTS}, **{T: "stress" for T in STRESS_SNAPSHOTS},
             **{T: "test" for T in TEST_SNAPSHOTS}}
censored = counts.drop(columns=["event", "non_event"]).sum(axis=1)
rates = pd.DataFrame({"used as": counts.index.map(period_of), "at-risk loans": counts.sum(axis=1),
                      "events": counts.event,
                      "event rate % (uncensored)": (100 * counts.event / (counts.event + counts.non_event)).round(2),
                      "censored %": (100 * censored / counts.sum(axis=1)).round(1)})
display(rates)

**The December 2019 spike.** The window of the December 2019 snapshot is January–December 2020, the first year of
the COVID-19 pandemic. Under the CARES Act (March 2020), many borrowers entered payment **forbearance** (the
`borrower_assistance_plan` field shows `F`). The delinquency status follows the due date of the last paid installment
(guide p. 10), so paused payments age into 30, 60 and 90 days past due, and these loans count as events under our
rule. The next two cells check this on the data: almost all December 2019 events had a forbearance flag in their
window, and their first 90+ month clusters in mid-2020.

This is a different mechanism from the ordinary credit deterioration the model is meant to learn. That is why
December 2019 and December 2020 are used **only as a stress test** (reported, never used to fit or to test), and why
training windows end by December 2019.

**What this cell does:** For the events of December 2018, 2019 and 2020, counts how many had a forbearance flag (`F`) or a disaster-related delinquency flag (`Y`) in their outcome window.

**Why it matters:** It tests the forbearance explanation of the spike with data, instead of only asserting it.

**What to look for:** 201912: 1,120 of the 1,199 events had a forbearance flag. 201812 (pre-pandemic): only 16 of 160.

In [ ]:
forbearance = []
for T in [201812, 201912, 202012]:                 # the last training year and the two stress years
    t_T = int(ym_index([T])[0])
    ev = snaps.loc[(snaps.snapshot == T) & (snaps.label == "event"), "loan_id"]
    w = perf[(perf.t > t_T) & (perf.t <= t_T + HORIZON) & perf.loan_id.isin(ev)]       # their outcome windows
    forbearance.append({"snapshot": T, "events": len(ev),
                        "with forbearance (F) in window": w.loc[w.borrower_assistance_plan.eq("F"), "loan_id"].nunique(),
                        "with disaster flag (Y) in window": w.loc[w.delinq_due_to_disaster.eq("Y"), "loan_id"].nunique()})
display(pd.DataFrame(forbearance).set_index("snapshot"))

**What this cell does:** For the December 2019 events, finds the first month in which each loan was 90+.

**Why it matters:** If forbearance drives the spike, the first 90+ months should bunch up a few months after the start of the pandemic, instead of spreading evenly over the year.

**What to look for:** Few loans in January–April 2020, then a peak in June 2020 (530 loans) and July 2020 (291).

In [ ]:
t19 = int(ym_index([201912])[0])
ev19 = snaps.loc[(snaps.snapshot == 201912) & (snaps.label == "event"), "loan_id"]
w19 = perf[(perf.t > t19) & (perf.t <= t19 + HORIZON) & perf.loan_id.isin(ev19)]
first90 = w19[w19.sev].groupby("loan_id").period.min().value_counts().sort_index()
display(first90.rename("Dec-2019 events: loans first 90+ in month").to_frame().T)

## 7. The train/test split

The split is made in **two** ways at once.

**By loan (hash split).** The same loan appears in several Decembers. If a loan were in both training and test, the
test would partly reward remembering that particular loan. So each loan is assigned to a pool once, for good:

$$ \text{bucket} = \text{SHA-256}(\text{loan ID}) \bmod 100, \qquad
   \text{train pool} \iff \text{bucket} < 70 $$

where 70 is `TRAIN_LOAN_SHARE`. SHA-256 is a fixed function: the same ID always gives the same 64-character hash, on any computer, in any run. So the
split is **reproducible without any random seed**, and adding or removing other loans never moves a loan to the
other pool. Because the hash looks random, about 70% of loans fall in the train pool.

**By time.** Training uses Decembers 2015–2018; testing uses Decembers 2021–2024. The model is judged on years it
has never seen, as it would be in real use (an out-of-time test). The pandemic windows (2019–2020) are left out of
both and only reported as a stress test (section 6).

**What this cell does:** Defines `loan_pool()`, the hash split described above.

**Why it matters:** This one line decides which loans may ever be used for training.

**What to look for:** Nothing is printed.

In [ ]:
def loan_pool(loan_id):
    """Deterministic, reproducible loan-level split (no randomness to seed)."""
    return int(hashlib.sha256(loan_id.encode()).hexdigest(), 16) % 100 < TRAIN_LOAN_SHARE

**What this cell does:** Shows the hash, the bucket and the pool for the five synthetic IDs.

**Why it matters:** It makes the hash split concrete on made-up IDs. Run the cell twice, or on another computer: the numbers never change.

**What to look for:** Each ID gets a bucket from 0 to 99; the pool is `True` exactly when the bucket is below 70.

In [ ]:
for loan in ["X01", "X02", "X03", "X04", "X05"]:
    h = hashlib.sha256(loan.encode()).hexdigest()       # 64 hex characters, always the same for the same ID
    print(f"{loan}: sha256 = {h[:12]}...  bucket = {int(h, 16) % 100:2d}  ->  train pool: {loan_pool(loan)}")

**What this cell does:** Adds the `in_train_pool` column to the snapshots and prints the share of loans in the train pool.

**Why it matters:** It checks that the hash split behaves as intended on the real loan IDs.

**What to look for:** About 70% of the distinct loans are in the train pool.

In [ ]:
snaps["in_train_pool"] = snaps.loan_id.map(loan_pool)
loans = snaps.drop_duplicates("loan_id")
print(f"{loans.in_train_pool.mean():.1%} of the {len(loans):,} distinct loans in the snapshots are in the train pool")

**Why censored rows are dropped before fitting.** A censored loan left the data (mostly by prepaying) before $t+12$
without a 90+ month. Its outcome is **unknown**: it might have gone 90+ later in the window, had it stayed. Counting
it as y = 0 would assume it would have stayed current; counting it as y = 1 would be wrong too. The draft outcome rules
therefore exclude these rows from fitting and evaluation, but count and report them (section 6). See
[CLAUDE.md → "Outcome specification"](../CLAUDE.md). Treating prepayment as y = 0 is a sensitivity analysis still
waiting for Dr. Ku's decision (exercise 4).

**What this cell does:** Keeps only uncensored rows, creates `y`, splits into train / test / stress, and asserts the two split rules.

**Why it matters:** These assertions are the guarantees behind every reported number: no loan is in both train and test, and every training date comes before every test date.

**What to look for:** `Train rows 101,843 (events 462) | Test rows 15,716 (events 119) | loan overlap = 0`.

In [ ]:
uncensored = snaps[snaps.label.isin(["event", "non_event"])].copy()          # drop censored rows (outcome unknown)
uncensored["y"] = (uncensored.label == "event").astype(int)                   # 1 = event, 0 = non_event
train = uncensored[uncensored.snapshot.isin(TRAIN_SNAPSHOTS) & uncensored.in_train_pool]    # early years, pool loans
test = uncensored[uncensored.snapshot.isin(TEST_SNAPSHOTS) & ~uncensored.in_train_pool]     # late years, other loans
stress = uncensored[uncensored.snapshot.isin(STRESS_SNAPSHOTS) & ~uncensored.in_train_pool] # pandemic, other loans
overlap = set(train.loan_id) & set(test.loan_id)
assert not overlap, f"{len(overlap)} loans in both train and test"
assert train.snapshot.max() < test.snapshot.min()                             # all train dates before all test dates
print(f"\nTrain rows {len(train):,} (events {train.y.sum()}) | Test rows {len(test):,} "
      f"(events {test.y.sum()}) | loan overlap = 0")

**What this cell does:** Summarizes each set: rows before and after dropping censored rows, events, event rate and distinct loans.

**Why it matters:** It shows how much data each step removes, and that test and stress have many fewer distinct loans than rows: the same loan appears in several Decembers, which matters for the bootstrap in section 9.

**What to look for:** Train 101,843 rows / 462 events (0.45%); test 15,716 / 119 (0.76%); stress 12,445 / 422 (3.39%).

In [ ]:
sets = {"train": (TRAIN_SNAPSHOTS, True), "test_2021_2024": (TEST_SNAPSHOTS, False),
        "stress_2019_2020": (STRESS_SNAPSHOTS, False)}
summary = []
for name, (dates, pool) in sets.items():
    before = snaps[snaps.snapshot.isin(dates) & (snaps.in_train_pool == pool)]
    kept = uncensored[uncensored.snapshot.isin(dates) & (uncensored.in_train_pool == pool)]
    summary.append({"set": name, "rows before dropping censored": len(before),
                    "censored rows dropped": len(before) - len(kept), "rows used": len(kept),
                    "events": int(kept.y.sum()), "event rate %": round(100 * kept.y.mean(), 3),
                    "distinct loans": kept.loan_id.nunique()})
display(pd.DataFrame(summary).set_index("set"))

## 8. The logistic regression model

For each row $i$ (one loan at one December), the model computes a score and turns it into a probability:

$$ z_i = b_0 + \mathbf{x}_i^{\top}\mathbf{b} \qquad\qquad p_i = \sigma(z_i) = \frac{1}{1 + e^{-z_i}} $$

and the coefficients are chosen to minimize the L2-penalized log-loss:

$$ \min_{b_0,\ \mathbf{b}} \;\; -\sum_{i=1}^{n} \Big[\, y_i \log p_i + (1 - y_i) \log (1 - p_i) \,\Big]
   \;+\; \frac{1}{2C}\, \lVert \mathbf{b} \rVert_2^2 $$

In plain words:

* $\mathbf{x}_i$: the row's features after preprocessing (standardized numbers and 0/1 category columns).
* $\mathbf{b}$: one coefficient per feature column; $b_0$: the intercept (the baseline log-odds). The intercept is
  not penalized.
* $z_i$: the **log-odds** of an event, $\log\frac{p_i}{1-p_i}$. $z = 0$ means $p = 0.5$; each $+1$ in $z$
  multiplies the odds by $e \approx 2.72$.
* $p_i$: the predicted probability that the loan becomes 90+ within 12 months.
* $y_i$: the observed label, 1 = event, 0 = non-event.
* The sum is the **log-loss**: it is small when $p_i$ is high for events and low for non-events.
* $\lVert \mathbf{b} \rVert_2^2 = \sum_j b_j^2$, and $C$ (`C = 1.0` in SETTINGS) sets how strongly large
  coefficients are penalized: a smaller $C$ means stronger shrinkage toward 0.

scikit-learn's `LogisticRegression` with the `lbfgs` solver minimizes exactly this (it multiplies the whole objective
by $C$, which does not change the solution). There is **no class weighting**, so $p_i$ stays on the real event-rate
scale (about 0.5%), which the Brier score and the calibration check need. Benchmark pair 2 plants a sign error in the
formula for $p_i$.

**What this cell does:** Lists the 16 numeric and 7 categorical features.

**Why it matters:** These lists are the model's input. To remove a feature, edit them here (see the section 13 table).

**What to look for:** Nothing is printed. `current_upb` is not in the lists: it is in the snapshot, but not a feature.

In [ ]:
numeric = ["fico", "orig_ltv", "orig_dti", "dti_missing", "orig_interest_rate", "log_orig_upb",
           "term_360", "multi_borrower", "mi_flag", "num_units", "loan_age", "max_dq_12m",
           "n_dq_months_12m", "ever_90_before_t", "ever_modified", "disaster_flag_t"]
categorical = ["status_t", "loan_purpose", "occupancy_status", "first_time_homebuyer",
               "property_type", "channel", "assist_plan_t"]

**The preprocessing pipeline.** Logistic regression needs complete numeric inputs on comparable scales:

* **Numeric features:** missing values (for example the 4,020 DTIs that were code 999) are filled with the
  **training median**; then each feature is standardized, $(x - \text{mean})/\text{sd}$, with the **training** mean
  and standard deviation. On standardized features, the L2 penalty treats every feature equally, and the
  coefficients become comparable (section 10).
* **Categorical features:** one-hot encoding gives one 0/1 column per level (for example `status_t_00`,
  `status_t_01`, `status_t_02`). Levels seen fewer than 20 times in training (`min_frequency=20`) are grouped into one
  `infrequent_sklearn` column. A level never seen in training gets all zeros (`handle_unknown="ignore"`).

**Why fit the preprocessing on training data only?** The medians, means, standard deviations and category lists are
part of the model. If they were computed on test data, information from the test years would leak into the model,
and in real use the future data would not exist yet. Putting everything in one `Pipeline` and calling `fit` only on
`train` guarantees this; `predict_proba` on test then *applies* the training values without refitting them.

**What this cell does:** Builds the preprocessing and the model as one scikit-learn `Pipeline`. The penalty `C` comes from SETTINGS (the reference uses `C = 1.0`).

**Why it matters:** Keeping preprocessing and model in one object makes it impossible to fit the preprocessing on the wrong data.

**What to look for:** Nothing is printed.

In [ ]:
pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),       # missing -> training median
                      ("scale", StandardScaler())]), numeric),            # (x - mean) / sd, from training
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), categorical)])   # 0/1 column per level
model = Pipeline([("pre", pre),
                  ("lr", LogisticRegression(C=C, max_iter=5000, random_state=SEED))])  # reference: C=1.0

**What this cell does:** Fits the pipeline on the training rows only.

**Why it matters:** This single line learns every number of the model: medians, means, standard deviations, categories and coefficients.

**What to look for:** The solver converges in far fewer iterations than `max_iter = 5000`. It takes a second or two.

In [ ]:
model.fit(train[numeric + categorical], train.y)          # preprocessing fitted on TRAIN only
print("lbfgs iterations:", model.named_steps["lr"].n_iter_[0], "(max_iter = 5000)")

**What this cell does:** Shows what the preprocessing learned from the training rows: the medians, means and standard deviations, the missing counts, and the category levels, including the ones grouped as infrequent.

**Why it matters:** These are the training-only statistics the text above talks about. They are applied unchanged to test and stress data.

**What to look for:** `orig_dti` has 9,193 missing values in the training rows, filled with the training median 35. Grouped as infrequent: the `9` of `first_time_homebuyer` (section 3) and the rare assistance plans `R` and `T`.

In [ ]:
num_pipe = model.named_steps["pre"].named_transformers_["num"]
learned = pd.DataFrame({"missing in train": train[numeric].isna().sum(),
                        "train median (fills missing)": num_pipe.named_steps["impute"].statistics_,
                        "train mean": num_pipe.named_steps["scale"].mean_,
                        "train sd": num_pipe.named_steps["scale"].scale_}, index=numeric)
display(learned.round(3))
ohe = model.named_steps["pre"].named_transformers_["cat"]
for col, levels, rare in zip(categorical, ohe.categories_, ohe.infrequent_categories_):
    print(f"{col:22s} levels: {list(levels)}   grouped as infrequent: {[] if rare is None else list(rare)}")

## 9. Evaluation

Three metrics, each answering a different question:

* **ROC-AUC (ranking).** Pick one loan that went 90+ and one that did not: the ROC-AUC is the probability that the
  model gave the bad loan the higher risk. 0.5 = coin flip, 1.0 = perfect ranking.
* **PR-AUC (average precision; finding the rare bad loans).** Summarizes, over all cut-offs, how many of the loans
  the model flags really go 90+ (precision). A model that guesses at random scores about the event rate (here under
  1%), so PR-AUC must be read against that base rate, not against 1.
* **Brier score (accuracy of the probabilities).** The mean squared gap between the predicted probability and what
  happened (0 or 1): $\text{Brier} = \frac{1}{n}\sum_i (p_i - y_i)^2$. Lower is better. It is compared with a
  "base-rate model" that predicts the set's own event rate for every loan. That comparison is generous to the
  base-rate model, because it uses the true event rate of the very set being scored.

**The loan-clustered bootstrap (95% confidence intervals).** The test set has 15,716 rows but only 4,429 distinct
loans (section 7 table), because a loan can appear in up to four test Decembers. Rows of the same loan are not independent: a weak borrower
tends to be weak in every year. Resampling **rows** would treat them as independent and give intervals that are too
narrow. The baseline resamples **loans** (with all their rows), `N_BOOT` times, recomputes the metric each time, and
reports the 2.5th and 97.5th percentiles. A resample that happened to contain no event at all would be skipped,
because the metric is undefined on it.

**What this cell does:** Defines `cluster_bootstrap_ci()`.

**Why it matters:** It produces the uncertainty ranges (95% CIs) that tell us whether a difference between two models is real.

**What to look for:** Nothing is printed. Note `rng = np.random.default_rng(seed)`: every call restarts from the same seed, so the CIs are exactly reproducible.

In [ ]:
def cluster_bootstrap_ci(df, metric, n=1000, seed=SEED):
    """95% CI resampling LOANS (a loan can appear in several test snapshots)."""
    rng = np.random.default_rng(seed)
    loans = df.loan_id.unique()
    groups = {k: v.index.values for k, v in df.groupby("loan_id")}      # loan -> its row positions
    stats = []
    for _ in range(n):
        pick = rng.choice(loans, size=len(loans), replace=True)         # draw loans with replacement
        idx = np.concatenate([groups[l] for l in pick])                 # all rows of the drawn loans
        y, p = df.y.values[idx], df.p.values[idx]
        if y.min() == y.max():                                          # no events (or no non-events): skip
            continue
        stats.append(metric(y, p))
    return [float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))]

**What this cell does:** Defines `evaluate()`. The bootstrap uses `n=N_BOOT` from SETTINGS (the reference used 1000).

**Why it matters:** It computes every number stored in `reference/metrics.json` for one set, and prints a two-line summary.

**What to look for:** Nothing is printed until the next cell runs it.

In [ ]:
def evaluate(df, name):
    y, p = df.y.values, df.p.values
    res = {"n": int(len(y)), "events": int(y.sum()), "event_rate": float(y.mean()),
           "mean_predicted": float(p.mean()),
           "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p)),
           "brier": float(brier_score_loss(y, p)),
           "brier_reference_base_rate": float(brier_score_loss(y, np.full_like(p, y.mean())))}
    res["roc_auc_95ci"] = cluster_bootstrap_ci(df, roc_auc_score, n=N_BOOT)            # reference: n=1000
    res["pr_auc_95ci"] = cluster_bootstrap_ci(df, average_precision_score, n=N_BOOT)
    print(f"\n[{name}] n={res['n']:,} events={res['events']} rate={res['event_rate']:.4f} "
          f"mean_p={res['mean_predicted']:.4f}")
    print(f"  ROC-AUC {res['roc_auc']:.3f} {np.round(res['roc_auc_95ci'], 3)} | "
          f"PR-AUC {res['pr_auc']:.3f} {np.round(res['pr_auc_95ci'], 3)} | "
          f"Brier {res['brier']:.5f} (base-rate model {res['brier_reference_base_rate']:.5f})")
    return res

**What this cell does:** Defines `calibration_table()`. It sorts loans into 10 equal-sized groups (deciles) by predicted risk and compares the mean prediction with the observed event rate in each group.

**Why it matters:** A model can rank well (high AUC) and still predict the wrong *level* of risk. Calibration checks the level.

**What to look for:** Nothing is printed.

In [ ]:
def calibration_table(df, bins=10):
    d = df.assign(bin=pd.qcut(df.p.rank(method="first"), bins, labels=False))   # decile 0 = lowest risk
    return d.groupby("bin").agg(n=("y", "size"), mean_predicted=("p", "mean"),
                                observed_rate=("y", "mean"), events=("y", "sum")).reset_index()

**What this cell does:** Scores the train, test and stress sets, evaluates each one (metrics + bootstrap CIs), and builds the test calibration table. Nothing is written here; with `SAVE_OUTPUTS = True`, the last cell of section 12 saves the files.

**Why it matters:** These are the headline results of the baseline.

**What to look for:** Test: ROC-AUC 0.894 [0.849, 0.930], PR-AUC 0.307, Brier 0.00620. Train ROC-AUC 0.890; stress 0.732. This is the slow cell; the time is printed at the end.

In [ ]:
start = time.perf_counter()
results, scored = {}, {}
for name, d in [("train", train), ("test_2021_2024", test), ("stress_2019_2020", stress)]:
    d = d.copy(); d["p"] = model.predict_proba(d[numeric + categorical])[:, 1]    # predicted P(event)
    d = d.reset_index(drop=True)                    # positions 0..n-1, used by the bootstrap's row lookup
    results[name] = evaluate(d, name)
    scored[name] = d                                # kept for the calibration table
cal = calibration_table(scored["test_2021_2024"])   # test calibration by decile (reference/calibration.csv)
print(f"\n(evaluation took {time.perf_counter() - start:.0f} s with N_BOOT = {N_BOOT})")

**What this cell does:** Collects the results of the three sets in one readable table.

**Why it matters:** It is easier to compare train, test and stress side by side than in the printed lines.

**What to look for:** Test PR-AUC 0.307 is about 40 times the test event rate (0.76%). Brier is below the base-rate model in all three sets. Mean predicted vs event rate: almost equal in train; lower than observed in test and stress.

In [ ]:
view = pd.DataFrame({name: {
    "rows": f"{r['n']:,}", "events": r["events"], "event rate": f"{r['event_rate']:.2%}",
    "mean predicted": f"{r['mean_predicted']:.2%}",
    "ROC-AUC [95% CI]": f"{r['roc_auc']:.3f} [{r['roc_auc_95ci'][0]:.3f}, {r['roc_auc_95ci'][1]:.3f}]",
    "PR-AUC [95% CI]": f"{r['pr_auc']:.3f} [{r['pr_auc_95ci'][0]:.3f}, {r['pr_auc_95ci'][1]:.3f}]",
    "Brier (base-rate model)": f"{r['brier']:.5f} ({r['brier_reference_base_rate']:.5f})"}
    for name, r in results.items()})
display(view)

**What this cell does:** Shows the test calibration table and plots it. The left panel is the original script's plot (saved as `outputs/calibration.png` when `SAVE_OUTPUTS = True`); the right panel zooms in on the 9 lowest deciles, which are squeezed into the corner of the left panel.

**Why it matters:** Points on the dashed diagonal mean the predicted risk equals the observed rate. Points above it mean the model predicts too little risk.

**What to look for:** 8 of the 10 points lie above the diagonal. The top decile: predicted about 3.6%, observed about 5.8%. In the zoom, the points jump up and down because each low decile holds only 0–10 events: that is noise, not a pattern.

In [ ]:
display(cal.round(5))
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.5))
for ax, part, title in [(axes[0], cal, "Calibration, test 2021-2024"),
                        (axes[1], cal[cal["bin"] < 9], "Zoom: the 9 lowest deciles")]:
    ax.plot(part.mean_predicted, part.observed_rate, "o-", color="black", label="model")
    m = max(part.mean_predicted.max(), part.observed_rate.max()) * 1.05
    ax.plot([0, m], [0, m], "--", color="grey", label="perfect")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed 90+ rate")
    ax.set_title(title); ax.legend(frameon=False); ax.grid(color="0.92"); ax.set_axisbelow(True)
fig.tight_layout(); plt.show()                      # shown only; section 12 saves the left panel if SAVE_OUTPUTS

**Why the model under-predicts on test (mean predicted 0.43% vs observed 0.76%), and why that is not a code
defect.** On the training rows, the mean prediction (0.456%) almost equals the observed rate (0.454%). That is a
property of logistic regression with an unpenalized intercept: at the optimum, the predictions sum to the number of
events in the training data. So the model's overall *level* is anchored to the 2015–2018 event rate. The test years
2021–2024 simply have a higher 90+ rate (section 6 table: about 0.67–0.90% per year, against 0.24–0.65% in the
training years). The population and the economic environment changed between the two periods: a **distribution
shift**. The ranking still holds up well (test ROC-AUC 0.894, about the same as on train), so the model separates
riskier from safer loans; only its level is off. The code does exactly what the specification says. As `CLAUDE.md`
states, distribution shift and imperfect accuracy are not implementation defects. Fixing the level (recalibrating on
recent data) would be a modelling change, not a bug fix, and is not part of the baseline.

## 10. Coefficients

**What this cell does:** Builds the coefficient table (reference copy: `reference/coefficients.csv`) and shows it sorted by size, with the odds ratio $e^{b}$ for the numeric features.

**Why it matters:** The coefficients show what the model has learned, and they are compared with the reference in section 12.

**What to look for:** 40 rows (39 feature columns + intercept). The largest are the `status_t` levels; among the numeric features: `fico` −0.375, `max_dq_12m` +0.332, `multi_borrower` −0.289, `orig_ltv` +0.256.

In [ ]:
names = model.named_steps["pre"].get_feature_names_out()
coefs = pd.DataFrame({"feature": names, "coef_standardized": model.named_steps["lr"].coef_[0]})
coefs.loc[len(coefs)] = ["intercept", model.named_steps["lr"].intercept_[0]]
coef_view = coefs.sort_values("coef_standardized", key=abs, ascending=False).set_index("feature").round(3)
coef_view["odds ratio per +1 sd"] = np.where(coef_view.index.str.startswith("num__"),
                                             np.exp(coef_view.coef_standardized).round(3), np.nan)
display(coef_view)

**How to read standardized coefficients.** Every numeric feature was standardized with the training mean and
standard deviation (sd). So a coefficient is the change in log-odds for a **one-sd increase** in that feature,
holding all other features fixed. Example: `fico` = −0.375 means one sd more FICO multiplies the odds of going 90+ by
$e^{-0.375} \approx 0.69$, a 31% reduction. Because all numeric features share this scale, their sizes can be
compared with each other. For a 0/1 flag, "one sd" is a fraction of the 0 → 1 step, so the full 0 → 1 effect is
$b / \text{sd}$, with the sd from section 8. Coefficients are associations, given the other features, not causes.
Correlated features share their effect: `max_dq_12m`, `n_dq_months_12m` and `status_t` all describe recent
delinquency, so the small negative values of `n_dq_months_12m` and `ever_90_before_t` should not be read on their
own. `CLAUDE.md` is explicit that odd-looking coefficients are not implementation defects.

**The one-hot caveat: compare levels with each other, not with zero.** The encoder keeps **all** levels of each
categorical variable (no reference level is dropped). Each row has exactly one level set to 1, so the levels of one
variable always add up to the intercept's column of 1s. The model could shift any amount between the intercept and
the levels without changing a single prediction, and the L2 penalty decides how it is split. A single level's
coefficient compared with 0 therefore means nothing (for example, all `assist_plan_t` levels are negative). Only
**differences between levels of the same variable** are meaningful. The next cell computes them for `status_t`.
Exercise 3 drops a reference level instead.

**What this cell does:** Computes, for each `status_t` level, its difference from the `00` (current) level and the matching odds ratio.

**Why it matters:** This is the correct way to read one-hot coefficients, and the status at $t$ is the strongest predictor in the model.

**What to look for:** `01` vs `00`: about +1.85 (odds about 6× higher); `02` vs `00`: about +3.04 (odds about 21× higher), other features held fixed.

In [ ]:
coef = coefs.set_index("feature").coef_standardized
status_levels = coef[coef.index.str.startswith("cat__status_t_")]
base = coef["cat__status_t_00"]                                   # the current (00) level as the comparison
display(pd.DataFrame({"coefficient": status_levels.round(3),
                      "difference vs status 00": (status_levels - base).round(3),
                      "odds ratio vs status 00": np.exp(status_levels - base).round(2)}))

## 11. Leakage check

**Future invariance, in plain words.** A feature at prediction month $t$ must be something you could actually have
known on 31 December of that year. If that is true, then **deleting or changing anything after $t$ cannot change any
feature**. If some feature does change, that feature has looked into the future: **leakage**. Leakage makes a model
look much better in a backtest than it can ever be in real use. The **label**, by contrast, *must* change, because it
is defined on months $t+1 \dots t+12$.

The checks below run on the toy panel from section 5, in two ways: **truncation** (delete every month after $t$) and
**perturbation** (keep the months after $t$ but change their values). Each time, rerun `build_snapshot()` and compare
the features with the original ones.

**What this cell does:** Builds the toy snapshot twice, once from the full panel and once from the panel with every month after $t$ deleted, and compares all feature columns.

**Why it matters:** This is the general leakage test. It does not need to know *how* a leak might happen: any feature that reads the future fails it.

**What to look for:** `Same loans: True` and `Feature columns identical ...: True`. The labels do change: with no future, every loan becomes `censored_end_of_data`.

In [ ]:
t_toy = int(ym_index([T_TOY])[0])
full = build_snapshot(toy, T_TOY).set_index("loan_id").sort_index()                  # all 30 months
cut = build_snapshot(toy[toy.t <= t_toy], T_TOY).set_index("loan_id").sort_index()   # months after t deleted
feature_cols = [c for c in full.columns if c not in ("label", "snapshot")]           # the label may use the future
print("Same loans:", list(full.index) == list(cut.index))
print("Feature columns identical after deleting every month after t:", full[feature_cols].equals(cut[feature_cols]))
display(pd.DataFrame({"label (full panel)": full.label, "label (months after t deleted)": cut.label}))

**What this cell does:** Builds the toy snapshot from a *perturbed* panel: every month after $t$ is kept, but its delinquency status is changed to `05` (150–179 days late), and its modification, disaster and assistance fields are set to `Y`, `Y` and `F`. `dq` and `sev` are recomputed with the same lines as `load()`. All feature columns are compared with the toy snapshot from the original panel.

**Why it matters:** Truncation shows that *deleting* the future changes no feature. Perturbation shows that *changing* it changes none either: a feature that read the future would pick up the new values. This is test T2 of `tests/test_baseline_no_leakage.py`, run on the toy panel.

**What to look for:** `Same loans: True` and `Feature columns identical ...: True`. The labels do change: every at-risk loan now has a 90+ month in its window, so all four become `event` (`X03` too: its records 201801–201804 lie after $t$).

In [ ]:
future = toy.t > t_toy                                         # every record after the prediction month
perturbed = toy.copy()
perturbed.loc[future, "delinq_status"] = "05"                  # 150-179 days late in every month after t ...
perturbed.loc[future, "modification_flag"] = "Y"               # ... and modified, disaster-flagged, in forbearance
perturbed.loc[future, "delinq_due_to_disaster"] = "Y"
perturbed.loc[future, "borrower_assistance_plan"] = "F"
dq_p = pd.to_numeric(perturbed.delinq_status, errors="coerce")  # recompute dq and sev, as load() does
perturbed["dq"] = dq_p
perturbed["sev"] = (dq_p >= 3) | (perturbed.delinq_status == "RA")
pert = build_snapshot(perturbed, T_TOY).set_index("loan_id").sort_index()
print("Same loans:", list(full.index) == list(pert.index))
print("Feature columns identical after changing every month after t:", full[feature_cols].equals(pert[feature_cols]))
display(pd.DataFrame({"label (original panel)": full.label, "label (months after t changed)": pert.label}))

**What this cell does:** A demonstration of the power check: makes a deliberately leaky copy of `build_snapshot()` (the 12-month lookback is taken from all records instead of only those up to $t$) and runs the same comparison on it.

**Why it matters:** A test that can never fail proves nothing. The earlier baseline had exactly such a check (an assertion that could never fail). Showing that the comparison catches a real leak is what makes its "identical" result above meaningful.

**What to look for:** `identical after truncation: False`. `X02` and `X05` get `max_dq_12m` = 3 from the full panel, because the leaky lookback sees their 90+ months after $t$.

In [ ]:
OLD = "h12 = hist[(hist.t > t - 12) & hist.loan_id.isin(ids)]"    # lookback from hist (months <= t)
NEW = "h12 = perf[(perf.t > t - 12) & perf.loan_id.isin(ids)]"    # same window, no upper bound: a leak
source = inspect.getsource(build_snapshot)
if source.count(OLD) != 1:
    print("build_snapshot() was edited, so this demonstration's text mutation no longer applies; skipped.")
else:
    namespace = dict(globals())
    exec(source.replace(OLD, NEW), namespace)                     # defines a leaky copy of the function
    leaky = namespace["build_snapshot"]
    lf = leaky(toy, T_TOY).set_index("loan_id").sort_index()[feature_cols]
    lc = leaky(toy[toy.t <= t_toy], T_TOY).set_index("loan_id").sort_index()[feature_cols]
    print("Leaky copy, features identical after truncation:", lf.equals(lc))
    cols = ["max_dq_12m", "n_dq_months_12m"]
    display(pd.concat({"leaky copy, full panel": lf[cols], "leaky copy, months after t deleted": lc[cols]}, axis=1))

**The formal tests.** `tests/test_baseline_no_leakage.py` runs these checks on this notebook's own `build_snapshot()`.
It reads the two code cells tagged `leakage-test-source` (the cells that define `ym_index()` in section 2 and
`build_snapshot()` in section 5) and runs them in a fresh Python namespace that holds only `np`, `pd` and
`HORIZON = 12` (the outcome window of the specification; in this notebook `HORIZON` comes from the SETTINGS cell).
It then tests the result on eight synthetic loans `T01`–`T08`: T1 truncation, T2 perturbation (changing status,
modification, disaster and assistance fields after $t$), T3 the mutation power check shown above (applied to the text
of the tagged cell), and T4 hand-computed feature values. Run them from the repository root:

```
uv run pytest tests -v
```

All 4 tests should pass.

**If you edit the two tagged cells**, keep their tag (it is stored in the cell's metadata) and keep them
self-contained: they may use only `np`, `pd`, `HORIZON` and each other. If no cell carries the tag, or the line
`h12 = hist[(hist.t > t - 12) & hist.loan_id.isin(ids)]` that T3 mutates is changed, the tests fail with a message
that says so.

## 12. Does the notebook match the reference?

The official reference results are in `reference/`. They were produced by the original script
(`src/baseline_logreg.py` at commit `9253e97`; see `reference/README.md`). This section compares the notebook's
results with them:

* **label counts** (`label_counts.csv`) and **set sizes** (rows and events per set): must be exactly equal;
* **metrics** (`metrics.json`: event rates, mean predictions, ROC-AUC, PR-AUC, Brier, and the CI bounds) and the
  run **configuration**: numbers must match to within $10^{-9}$;
* **coefficients** (`coefficients.csv`) and the **calibration table** (`calibration.csv`): within $10^{-9}$.

It never stops with an error. If something differs, it prints `DIFFERS FROM THE REFERENCE` and a table of every
difference (reference = old, notebook = new). That is the expected result after you have deliberately changed
something. With `N_BOOT` other than 1000, only the confidence-interval items (`*_95ci`) should differ.

The last cell writes the result files to `outputs/` when `SAVE_OUTPUTS = True`.

**What this cell does:** Adds the run configuration and the library versions to `results`, in the same form as `reference/metrics.json`.

**Why it matters:** The configuration is part of the reference, so a changed setting appears in the difference table by name (for example `config.horizon_months: 12 → 6`).

**What to look for:** The printed configuration matches the SETTINGS cell.

In [ ]:
results["config"] = {"seed": SEED, "train_snapshots": TRAIN_SNAPSHOTS, "test_snapshots": TEST_SNAPSHOTS,
                     "horizon_months": HORIZON, "train_loan_share_pct": TRAIN_LOAN_SHARE,
                     "C": C, "features_numeric": numeric, "features_categorical": categorical}
results["versions"] = {"python": platform.python_version(), "numpy": np.__version__,
                       "pandas": pd.__version__, "scikit-learn": sklearn.__version__}
print({k: v for k, v in results["config"].items() if not k.startswith("features")})

**What this cell does:** Defines small helpers: `flatten()` turns nested results into named items (for example `test_2021_2024.roc_auc`), and `differences()` lists every item that differs by more than the tolerance or exists on only one side.

**Why it matters:** Comparing item by item, rather than whole files, tells you exactly *what* changed, which is the point of this section.

**What to look for:** Nothing is printed.

In [ ]:
TOL = 1e-9                                   # numbers closer than this count as equal
REF_DIR = Path("reference")                  # official reference results (read only; see reference/README.md)

def flatten(obj, prefix=""):
    """{'train': {'roc_auc_95ci': [a, b]}} -> {'train.roc_auc_95ci[0]': a, 'train.roc_auc_95ci[1]': b}"""
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            out.update(flatten(v, f"{prefix}.{k}" if prefix else str(k)))
        return out
    if isinstance(obj, list) and obj and all(isinstance(v, float) for v in obj):
        return {f"{prefix}[{i}]": v for i, v in enumerate(obj)}       # a CI -> its two bounds
    return {prefix: obj}                                              # a number, a text, or a list of names/dates

def is_number(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool)

def differences(part, ref, new, missing="(not present)"):
    rows = []
    for key in list(ref) + [k for k in new if k not in ref]:
        r, n = ref.get(key, missing), new.get(key, missing)
        numeric_pair = is_number(r) and is_number(n)
        if not (abs(r - n) <= TOL if numeric_pair else r == n):
            rows.append({"part": part, "item": key, "reference (old)": r, "notebook (new)": n,
                         "abs. difference": abs(r - n) if numeric_pair else None})
    return rows

**What this cell does:** Reads the four reference files and compares them item by item with the notebook's results, collecting every difference in the table `diff`.

**Why it matters:** Each part (set sizes, metrics, configuration, label counts, coefficients, calibration) is compared separately, so a difference can be traced to one step of the pipeline.

**What to look for:** Nothing is printed; the verdict follows in the next cell.

In [ ]:
ref_metrics = json.loads((REF_DIR / "metrics.json").read_text())
ref_counts = pd.read_csv(REF_DIR / "label_counts.csv", index_col="snapshot")
ref_coefs = pd.read_csv(REF_DIR / "coefficients.csv")
ref_cal = pd.read_csv(REF_DIR / "calibration.csv")

def part_of(key):
    return "set sizes" if key.split(".")[-1] in ("n", "events") else "config" if key.startswith("config.") else "metrics"
def count_items(c):          # a missing label-count cell means 0 loans
    return {f"{s} / {lab}": int(v) for (s, lab), v in c.stack().items() if v != 0}
def calibration_items(c):
    return {f"bin {int(b)} / {col}": c.loc[i, col] for i, b in c["bin"].items() for col in c.columns if col != "bin"}

ref_flat = flatten({k: v for k, v in ref_metrics.items() if k != "versions"})
new_flat = flatten({k: v for k, v in results.items() if k != "versions"})
checks = [(p, {k: v for k, v in ref_flat.items() if part_of(k) == p},
              {k: v for k, v in new_flat.items() if part_of(k) == p}, "(not present)") for p in ["set sizes", "metrics", "config"]]
checks += [("label counts", count_items(ref_counts), count_items(counts), 0),
           ("coefficients", dict(zip(ref_coefs.feature, ref_coefs.coef_standardized)),
                            dict(zip(coefs.feature, coefs.coef_standardized)), "(not present)"),
           ("calibration", calibration_items(ref_cal), calibration_items(cal), "(not present)")]
rows, checked = [], {}
for part, ref, new, missing in checks:
    rows += differences(part, ref, new, missing)
    checked[part] = len(set(ref) | set(new))
diff = pd.DataFrame(rows, columns=["part", "item", "reference (old)", "notebook (new)", "abs. difference"])

**What this cell does:** Prints the verdict, the number of values compared and differing in each part, and, if anything differs, the full table of differences (reference = old, notebook = new).

**Why it matters:** This is the proof that the notebook reproduces the reference exactly, and the place that tells you what your own edits changed.

**What to look for:** `MATCHES THE REFERENCE MODEL`, with 0 differences in every part (6 set sizes, 30 metrics, 8 configuration items, 40 label counts, 40 coefficients, 40 calibration values).

In [ ]:
print("=" * 60)
print("MATCHES THE REFERENCE MODEL" if diff.empty else "DIFFERS FROM THE REFERENCE")
print("=" * 60)
display(pd.DataFrame({"values compared": checked,
                      "differences": diff.part.value_counts().reindex(list(checked), fill_value=0)}))
if not diff.empty:
    with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
        display(diff)
if N_BOOT != 1000:
    print(f"Note: N_BOOT = {N_BOOT}, but the reference used 1000 resamples, so the *_95ci items are expected to differ.")
if results["versions"] != ref_metrics.get("versions"):
    print("Note: library versions differ from the reference run:", ref_metrics.get("versions"), "->", results["versions"])

**What this cell does:** Only if `SAVE_OUTPUTS = True` (SETTINGS cell): writes the five result files to `outputs/` (gitignored) in the original script's format: `metrics.json`, `label_counts.csv`, `coefficients.csv`, `calibration.csv` and `calibration.png` (the single-panel calibration plot). With the default `False` it writes nothing.

**Why it matters:** The first four files have exactly the format of the files in `reference/`; with the default settings they are byte-identical to them. They are the files to review and copy into `reference/` if a change is ever made official (section 13). The notebook itself never writes to `reference/`.

**What to look for:** `SAVE_OUTPUTS = False: nothing written.` With `True`, the folder and the five file names.

In [ ]:
if SAVE_OUTPUTS:
    out = Path("outputs"); out.mkdir(exist_ok=True)                  # gitignored; reference/ is never written
    counts.to_csv(out / "label_counts.csv")
    cal.to_csv(out / "calibration.csv", index=False)
    coefs.to_csv(out / "coefficients.csv", index=False)
    (out / "metrics.json").write_text(json.dumps(results, indent=2))  # metrics, config and versions
    fig, ax = plt.subplots(figsize=(4.5, 4.5))                        # the original script's single-panel plot
    ax.plot(cal.mean_predicted, cal.observed_rate, "o-", color="black", label="model")
    m = max(cal.mean_predicted.max(), cal.observed_rate.max()) * 1.05
    ax.plot([0, m], [0, m], "--", color="grey", label="perfect")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed 90+ rate")
    ax.set_title("Calibration, test 2021-2024"); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(out / "calibration.png", dpi=150); plt.close(fig)   # saved, not shown
    print(f"Saved to {out.resolve()}: metrics.json, label_counts.csv, coefficients.csv, calibration.csv, calibration.png")
else:
    print("SAVE_OUTPUTS = False: nothing written.")

## 13. Where to make changes, and optional exercises

### Safe edit points

| What to change | Where | Rerun sections | Reference numbers that will change (section 12) |
|---|---|---|---|
| Snapshot dates (`TRAIN_SNAPSHOTS`, `TEST_SNAPSHOTS`, `STRESS_SNAPSHOTS`) | SETTINGS cell (§1) | 6 → 12 | label counts (rows added or removed), set sizes, every metric and CI, coefficients, calibration, `config.*_snapshots`. Keep every window inside the data: December + `HORIZON` months ≤ 202603. |
| Horizon (`HORIZON`) | SETTINGS cell (§1) | 5 → 12 (the toy uses it too) | every label, so label counts, set sizes, metrics, coefficients, calibration, `config.horizon_months`. |
| Penalty (`C`) | SETTINGS cell (§1) | 8 → 12 | coefficients, every metric and CI, calibration, `config.C`. Label counts and set sizes stay the same. |
| Bootstrap size (`N_BOOT`) or `SEED` | SETTINGS cell (§1) | 9 → 12 | only the `*_95ci` items (and `config.seed` for `SEED`). |
| Save the result files (`SAVE_OUTPUTS`) | SETTINGS cell (§1) | the last cell of 12 | none: it only writes `outputs/`. |
| Remove a feature | the `numeric` / `categorical` lists (§8, first cell) | 8 → 12 | coefficients (its row disappears), metrics, calibration, `config.features_*`. |
| Add a feature | compute it in `origination_features()` (§3, and add it to `keep`) or in `build_snapshot()` (§5, from `hist`/`now` only, i.e. months ≤ $t$); then add its name to `numeric` or `categorical` (§8) | from the section you edited → 12, and rerun §11 and `uv run pytest tests -v` to confirm there is no leakage | coefficients (a new row), metrics, calibration, `config.features_*`. |
| Censoring policy: prepayment as y = 0 (sensitivity run) | the `uncensored = ...` line (§7, the split cell) | 7 → 12 | set sizes (more rows, same events), event rates, metrics, coefficients, calibration. Label counts stay the same (they are counted before this step). |

When in doubt, **Restart** the kernel and **Run All**. Functions read settings such as `HORIZON` when they *run*, but
`cluster_bootstrap_ci()` stores `SEED` when it is *defined*, so rerunning only part of the notebook can mix old and
new values.

### Optional exercises (not applied; each is a change you can try)

1. **Map categorical "9"/"99" codes to an explicit "unknown" category.**
   *Hint:* in `origination_features()`, after the sentinel loop, set
   `o.loc[o[c].isin(["9", "99"]), c] = "unknown"` for the five categorical fields of section 3. Check with
   `value_counts()` that only the 8 `first_time_homebuyer` loans change. Then ask yourself: with
   `min_frequency=20`, does the encoder treat "unknown" any differently from before?
2. **Turn current balance into a feature: current UPB ÷ original UPB, using only month-$t$ data.**
   *Hint:* `build_snapshot()` already copies `current_upb` from the month-$t$ record (`now`), so it uses no future
   data. Add `orig_upb` to `keep` in `origination_features()`, compute
   `snaps["upb_ratio"] = snaps.current_upb / snaps.orig_upb` after the merge in §6, add `"upb_ratio"` to `numeric`,
   and rerun §6–12. Look at `describe()` first: the guide (p. 10) says current UPB is rounded to the nearest $1,000
   when the loan age is 6 months or less, and modified loans can carry deferred balances.
3. **Use one-hot encoding with one reference level dropped, for readable coefficients.**
   *Hint:* `OneHotEncoder(handle_unknown="ignore", min_frequency=20, drop="first")`. Each remaining level's
   coefficient is then a difference from the dropped level (the first in sorted order, for example `status_t` `00`).
   Compare with the differences computed in §10. Read scikit-learn's documentation on how `drop` interacts with
   `handle_unknown="ignore"` and `min_frequency` before trusting the result.
4. **Count prepayment as y = 0 and compare event rates and AUC.** This is one of the outcome decisions waiting for
   Dr. Ku.
   *Hint:* in §7, use `snaps.label.isin(["event", "non_event", "censored_prepay"])`. `y` stays
   `label == "event"`, so prepaid loans become 0. Compare the new train/test event rates and ROC-AUC with the
   reference values (section 12 lists them), and think about which direction you expect the event rate to move,
   and why.

> **Reminder: making a change official.** This notebook is the baseline, and `reference/` holds its official
> results; experimenting here changes neither. To make a change official, do it in a separate, reviewed step: edit the
> notebook (and `tests/test_baseline_no_leakage.py` if `build_snapshot()` changes), Restart and Run All with
> `SAVE_OUTPUTS = True`, check the differences that section 12 reports, copy the four files from `outputs/` into
> `reference/`, update `CLAUDE.md`, and run `uv run pytest tests -v`.